# 08 - Hanoi noise model

**Direct training on our measurements, with honest validation.**

Ce notebook :
1. computes the OSM morphology features (300 m radius) for our measurement points;
2. runs the full evaluation - **600 m spatial blocks, baselines, ablation, bootstrap CIs** -
   via `scripts/04_evaluate_models.py` ;
3. trains the final model on all points and saves it;
4. renvoie vers `scripts/07_export_gama_inputs.py` pour produire la carte.

### What changed (methodological correction, August 2026)

- **The CV grouped on `lat/lon.round(3)` (~110 m) is removed.** The features are
  aggregates over a disc of **radius 300 m**: two points 110 m apart share more than
  85 % of their disc. The model saw near-twins of its test points, and the
  R2 0.45 that was announced was not out-of-sample (Roberts et al. 2017, *Ecography*).
  Protocoles retenus : **block-CV 600 m**, **buffered leave-one-out (300 m)**, **leave-one-site-out**.
- **The "Bach Khoa" grid is removed.** It covered a district **with no measurement at all**,
  predicted by a model whose leave-one-site-out is negative. The map is now produced
  by `scripts/07_export_gama_inputs.py`, **over the envelope actually sampled** (3 sites + 400 m).
- **Baselines and an ablation are added** to measure the net contribution of morphology.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
# feature and path code now lives in the installed package (pip install -e .)

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

CRS_HANOI = 'EPSG:32648'   # UTM 48N
R = 300                    # rayon des features de morphologie (m)
AREA_M2 = np.pi * R ** 2
MEASURES = '../data/processed/measurements.csv'
PROC = '../data/interim'
os.makedirs(PROC, exist_ok=True)
os.makedirs('../models', exist_ok=True)

hanoi = pd.read_csv(MEASURES, parse_dates=['timestamp'])
hanoi['hour'] = hanoi.timestamp.dt.hour
hanoi['is_weekend'] = hanoi.timestamp.dt.dayofweek.isin([5, 6]).astype(int)
hanoi = hanoi.dropna(subset=['noise_dB', 'latitude', 'longitude']).reset_index(drop=True)
print(f'{len(hanoi)} measurements - sites {hanoi.site.value_counts().to_dict()}')
print(f'dB {hanoi.noise_dB.min():.0f}-{hanoi.noise_dB.max():.0f} · sd {hanoi.noise_dB.std():.1f}')

## 1. OSM cache around the measurement points

Downloaded once then reread. This cache is what is then consumed by
`04_evaluate_models.py` and `07_export_gama_inputs.py` - a single geographic source of truth.

In [ ]:
HB_PATH = f'{PROC}/hanoi_sites_buildings.gpkg'
HG_PATH = f'{PROC}/hanoi_sites_roads.graphml'
MARGIN = 0.006   # ~600 m, pour que les disques de 300 m des points de bord soient complets

if not (os.path.exists(HB_PATH) and os.path.exists(HG_PATH)):
    print('Downloading OSM around the measurement points (slow the first time)...')
    ox.settings.timeout = 600
    bbox = (hanoi.longitude.min() - MARGIN, hanoi.latitude.min() - MARGIN,
            hanoi.longitude.max() + MARGIN, hanoi.latitude.max() + MARGIN)
    hb = ox.features_from_bbox(bbox, tags={'building': True})
    hb = hb[hb.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
    hb[['geometry']].to_file(HB_PATH, driver='GPKG')
    ox.save_graphml(ox.graph_from_bbox(bbox, network_type='drive'), HG_PATH)

from noise_hanoi import features as egz     # load_osm() / morphology(): shared logic, not duplicated
_, bld_c, nodes, edges = egz.load_osm()
print(f'{len(bld_c)} buildings - {len(edges)} road segments in cache')

## 2. Features de morphologie (rayon 300 m) pour nos points

In [ ]:
pts = gpd.GeoDataFrame(hanoi, geometry=gpd.points_from_xy(hanoi.longitude, hanoi.latitude),
                       crs='EPSG:4326').to_crs(CRS_HANOI)
feats = egz.morphology(pts, bld_c, nodes, edges)

MORPHO   = ['built_area_ratio', 'road_density_km_km2', 'intersection_count', 'dist_road_m']
TIME     = ['hour', 'is_weekend']
FEATURES = MORPHO + TIME

df = pd.concat([hanoi, feats], axis=1)
df['x'], df['y'] = pts.geometry.x.values, pts.geometry.y.values
# 600 m spatial block = 2 x the feature radius: the unit of the CV split.
df['block'] = (np.floor(df.x / 600).astype(int).astype(str) + '_' +
               np.floor(df.y / 600).astype(int).astype(str))
print(f'{df.block.nunique()} blocs spatiaux de 600 m pour {len(df)} points '
      f'(median {df.groupby("block").size().median():.0f} points/block)')
df[FEATURES].describe().round(2)

## 3. Honest evaluation - baselines, ablation, confidence intervals

Everything is in `scripts/04_evaluate_models.py` (single source, usable outside the notebook).
It evaluates **eight models on exactly the same splits**:

| | |
|---|---|
| `global_mean`, `site_mean` | planchers |
| `site_hour_mean` | lookup table (site, hour) - **the baseline to beat** |
| `dist_road`, `idw` | physique minimale, interpolation pure |
| `lgbm_time`, `lgbm_morpho` | ablations |
| `lgbm_full` | the project's model |

The line to read is **`morphology_gain`**: dR2 and dMAE of `lgbm_full` against `site_hour_mean`.
That is the numeric answer to "does urban morphology contribute anything?".

In [ ]:
!cd .. && python3 scripts/04_evaluate_models.py

In [ ]:
import json
metrics = json.load(open('../models/metrics.json'))

ref = metrics['meta']['headline_protocol']          # 'bloo' si disponible, sinon 'block_cv'
rows = [{'model': m['label'], 'R2': round(m['r2'], 3),
         'IC95 R²': f"[{m['r2_ci95'][0]:.2f}, {m['r2_ci95'][1]:.2f}]",
         'MAE': round(m['mae'], 2), 'r': round(m['r'], 2)}
        for m in metrics[ref]['models'].values()]
print(f"Reference protocol: {metrics[ref]['label']}")
display(pd.DataFrame(rows))

g = metrics[ref]['morphology_gain']
print(f"\nApport propre de la morphologie : ΔR² {g['delta_r2']:+.3f} · ΔMAE {g['delta_mae_dB']:+.2f} dB")
print('\nLeave-one-site-out (generalisation to an unseen typology):')
for s, v in metrics['loso_per_site'].items():
    print(f"  {s:18} n={v['n']:3}  R² {v['r2']:6.2f}  MAE {v['mae']:5.2f}")

## 4. Comparison: Uganda -> Hanoi transfer

Kept as a **methodological result** (see `docs/negative-results.md`), not as a
method. Evaluated on the same 600 m blocks as the rest, so the comparison is fair.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr

# text booster: portable across LightGBM versions,
# unlike the sklearn pickle, which required the exact same version.
uganda = lgb.Booster(model_file='../models/surrogate_lgbm_v2_uganda.txt')

# The Uganda model is applied as is, with an offset fitted on the training blocks
# of each fold only - otherwise the offset itself would leak.
oof = np.full(len(df), np.nan)
for tr, te in GroupKFold(5).split(df, df.noise_dB, df.block):
    off = (df.noise_dB.iloc[tr] - uganda.predict(df[FEATURES].iloc[tr])).mean()
    oof[te] = uganda.predict(df[FEATURES].iloc[te]) + off

print(f'Transfert Ouganda + offset (blocs 600 m) : '
      f'r {pearsonr(df.noise_dB, oof)[0]:.2f} | R² {r2_score(df.noise_dB, oof):.2f} | '
      f'MAE {mean_absolute_error(df.noise_dB, oof):.1f} dB')
print(f"To be compared with the direct model: R2 {metrics[ref]['models']['lgbm_full']['r2']:.2f}")

## 5. Final model and map

The final model is trained on **all** the points (no metric is drawn from it: the
published figures all come from section 3).

**The map is no longer produced here.** `scripts/07_export_gama_inputs.py` generates it over
the envelope actually sampled - the 3 sites + 400 m of margin - with one column per hour (h5...h21).
Predicting beyond that envelope would extrapolate to typologies that the
leave-one-site-out shows are not mastered.

In [ ]:
import lightgbm as lgb

final = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15,
                          min_child_samples=10, random_state=0, verbose=-1)
final.fit(df[FEATURES], df.noise_dB)
final.booster_.save_model('../models/surrogate_lgbm_hanoi_direct.txt')

imp = pd.Series(final.booster_.feature_importance('gain'), index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 3.2), title='Feature importance (gain) - final model')

print('Model saved -> models/surrogate_lgbm_hanoi_direct.txt')
print('\nSuite du pipeline :')
print('  python3 scripts/07_export_gama_inputs.py    # map over the 3 measured zones, 05:00-21:00')
print('  python3 scripts/08_validate_simulation.py  # grid <-> measurements check (in-sample)')
print('  python3 scripts/build_report.py         # rapport PDF (lit metrics.json)')